<a href="https://colab.research.google.com/github/nafisa-n01/Phishing-URL-Detection/blob/main/Phishing_URL_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# All imports here

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

## Loading the Dataset

Before running this notebook, set up the dataset in your own Google Drive so the folder structure looks like this:

```
MyDrive/
└── CSE445/
    └── PhiUSIIL_Phishing_URL_Dataset.csv
```

### Step 1: Add the dataset to your Google Drive
1. Open [Google Drive](https://drive.google.com).
2. In **My Drive**, create a folder named `CSE445`.
3. Open the `CSE445` folder and upload `PhiUSIIL_Phishing_URL_Dataset.csv` into it.

> Do this in your own Drive, with the exact same folder and file names, so the path below works for everyone.

### Step 2: Connect Colab to Google Drive
To load the data, we first need to connect this Colab notebook with Google Drive. Run the cell below and allow access when prompted.

In [2]:
# If you work with Google Colaboratory, please run this as well.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 3: Set the working directory
Move into the folder where the dataset is stored. After `%cd`, all file paths are relative to `CSE445`.

In [3]:
# Change the working directory to the project folder in Google Drive
%cd "/content/drive/MyDrive/CSE445"

/content/drive/MyDrive/CSE445


### Step 4: Verify the path
Run the cell below to check if the path is correctly set. If it prints "Some files are missing", re-check that the folder is named `CSE445` and the file name matches exactly.

In [4]:
from pathlib import Path

PATH = Path.cwd()

dataset_file = PATH / "PhiUSIIL_Phishing_URL_Dataset.csv"

# Check if the file exists
if dataset_file.exists():
    print("Dataset file exists and path is correctly set.")
else:
    print("Dataset file is missing or path is not correctly set.")

Dataset file exists and path is correctly set.


> ## Finally start here the imports are at the top of the notebook


## Data Cleaning

### Data overview

In [5]:
df = pd.read_csv("PhiUSIIL_Phishing_URL_Dataset.csv")

In [6]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 235795
Columns: 56


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   FILENAME                    235795 non-null  object 
 1   URL                         235795 non-null  object 
 2   URLLength                   235795 non-null  int64  
 3   Domain                      235795 non-null  object 
 4   DomainLength                235795 non-null  int64  
 5   IsDomainIP                  235795 non-null  int64  
 6   TLD                         235795 non-null  object 
 7   URLSimilarityIndex          235795 non-null  float64
 8   CharContinuationRate        235795 non-null  float64
 9   TLDLegitimateProb           235795 non-null  float64
 10  URLCharProb                 235795 non-null  float64
 11  TLDLength                   235795 non-null  int64  
 12  NoOfSubDomain               235795 non-null  int64  
 13  HasObfuscation

In [8]:
df.dtypes.value_counts()

,count
int64,41
float64,10
object,5


In [9]:
pd.set_option("display.max_columns", None)

In [10]:
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,558,9381,1,à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...,0.000000,0.000000,0,1,1,0,0,0,0,1,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,618,9381,1,johannes gutenberg-universitÃ¤t mainz,55.555556,55.555556,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,467,682,1,voice fm southampton,46.666667,46.666667,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,6356,26824,1,home page: seminars in fetal and neonatal medi...,0.000000,0.000000,0,1,1,0,0,0,1,12,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,6089,28404,1,fundaciÃ³n rewilding argentina,100.000000,100.000000,0,1,1,1,1,1,0,2,0,1,1,1,0,1,1,0,1,244,15,34,72,1,85,1


In [15]:
df["label"].unique()

array([1, 0])

In [20]:
df[df["label"] == 0].head(4)   # phishing

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
11,mw42508.txt,http://www.teramill.com,22,www.teramill.com,16,0,com,82.644628,1.000000,0.522907,0.067418,3,1,0,0,0.0,10,0.455,0,0.000,0,0,0,1,0.045,0,2,87,0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
20,mw16985.txt,http://www.f0519141.xsph.ru,26,www.f0519141.xsph.ru,20,0,ru,55.284926,0.923077,0.018013,0.015061,2,2,0,0,0.0,6,0.231,7,0.269,0,0,0,2,0.077,0,9,44,1,400 bad request,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
21,mw131249.txt,http://www.shprakserf.gq,24,www.shprakserf.gq,17,0,gq,56.551141,1.000000,0.000053,0.042982,2,1,0,0,0.0,12,0.500,0,0.000,0,0,0,1,0.042,0,20,211,1,shprakserf.gq,0.0,0.0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
27,7972389.txt,https://service-mitld.firebaseapp.com/,37,service-mitld.firebaseapp.com,29,0,com,64.645264,0.480000,0.522907,0.059401,3,1,0,0,0.0,26,0.703,0,0.000,0,0,0,3,0.081,1,16,113,0,0,0.0,0.0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [21]:
df[df["label"] == 1].head(4)   # legitimate

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,558,9381,1,à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...,0.000000,0.000000,0,1,1,0,0,0,0,1,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,618,9381,1,johannes gutenberg-universitÃ¤t mainz,55.555556,55.555556,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,467,682,1,voice fm southampton,46.666667,46.666667,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,6356,26824,1,home page: seminars in fetal and neonatal medi...,0.000000,0.000000,0,1,1,0,0,0,1,12,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31,1


> try checking first 1 phishing and 1 legitimate and compare the rows <br>
then try 3 <br>
then 6 <br>
you'll notice the differences and some possible outliers

In [22]:
df.select_dtypes(include="object").columns.tolist()

['FILENAME', 'URL', 'Domain', 'TLD', 'Title']

In [23]:
df.select_dtypes(include="object").nunique()

,0
FILENAME,235795
URL,235370
Domain,220086
TLD,695
Title,197874


for these 5 columns compare legitimate and phishing ones

In [25]:
text_cols = ["FILENAME", "URL", "Domain", "TLD", "Title"]

phishing = df[df["label"] == 0]
legit = df[df["label"] == 1]

print("Phishing rows:", len(phishing))
print("Legitimate rows:", len(legit))

Phishing rows: 100945
Legitimate rows: 134850


In [26]:
pd.DataFrame({
    "phishing_unique": phishing[text_cols].nunique(),
    "legit_unique": legit[text_cols].nunique(),
    "phishing_rows": len(phishing),
    "legit_rows": len(legit),
})

,phishing_unique,legit_unique,phishing_rows,legit_rows
FILENAME,100945,134850,100945,134850
URL,100520,134850,100945,134850
Domain,85290,134850,100945,134850
TLD,556,473,100945,134850
Title,65131,132965,100945,134850


In [30]:
phishing[text_cols].head(4)

,FILENAME,URL,Domain,TLD,Title
11,mw42508.txt,http://www.teramill.com,www.teramill.com,com,0
20,mw16985.txt,http://www.f0519141.xsph.ru,www.f0519141.xsph.ru,ru,400 bad request
21,mw131249.txt,http://www.shprakserf.gq,www.shprakserf.gq,gq,shprakserf.gq
27,7972389.txt,https://service-mitld.firebaseapp.com/,service-mitld.firebaseapp.com,com,0


In [31]:
legit[text_cols].head(4)

,FILENAME,URL,Domain,TLD,Title
0,521848.txt,https://www.southbankmosaics.com,www.southbankmosaics.com,com,à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...
1,31372.txt,https://www.uni-mainz.de,www.uni-mainz.de,de,johannes gutenberg-universitÃ¤t mainz
2,597387.txt,https://www.voicefmradio.co.uk,www.voicefmradio.co.uk,uk,voice fm southampton
3,554095.txt,https://www.sfnmjournal.com,www.sfnmjournal.com,com,home page: seminars in fetal and neonatal medi...


In [33]:
page_cols = [
    "HasExternalFormSubmit", "HasSocialNet", "HasSubmitButton",
    "HasHiddenFields", "HasPasswordField", "Bank", "Pay", "Crypto",
    "HasCopyrightInfo", "NoOfImage", "NoOfCSS", "NoOfJS",
    "NoOfSelfRef", "NoOfEmptyRef", "NoOfExternalRef"
]

In [34]:
phishing[page_cols].head(4)

,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
20,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
27,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [35]:
legit[page_cols].head(4)

,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
0,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124
1,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217
2,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5
3,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31


to check random rows


In [36]:
phishing[page_cols].sample(4, random_state=42)

,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
170643,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
229388,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
189747,0,0,0,0,1,0,0,0,1,3,0,6,0,0,1
39922,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [37]:
legit[page_cols].sample(4, random_state=42)

,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
157668,0,1,0,1,0,0,0,0,1,30,4,3,17,0,10
190150,0,0,0,1,0,0,1,0,0,77,2,10,82,0,79
116677,0,1,1,1,0,0,1,0,0,16,2,6,24,7,32
42508,1,1,1,1,0,0,1,0,1,49,58,75,163,7,161


#### Observations
- Shape: 235,795 rows × 56 columns
- Types: 41 int, 10 float, 5 text columns
- No missing values shown in `df.info()`
- Target `label`: 1 = legitimate, 0 = phishing (100,945 phishing, 134,850 legitimate)
- Phishing rows mostly have 0 or tiny values in the page-content columns (images, JS, references), while legit rows have real, varied counts
- Phishing `Title` often holds `0` or `400 bad request`, and some legit titles have garbled characters

#### Things to flag
- `FILENAME`: unique per row, so it is an identifier. 3 of 4 phishing filenames viewed start with `mw`, so check for label leakage
- `URL`: 425 duplicate URLs, all in the phishing class
- `Domain`, `Title`: raw text with very high cardinality
- `TLD`: 695 categories, the only possible categorical candidate
- Many zeros in phishing page-content columns, which may mean the pages were empty when collected
- Still to verify: whether `http` vs `https` in `URL` is already captured by `IsHTTPS`, and whether `Title` adds anything beyond `HasTitle` and the match scores

## Exploratory Data Analysis (EDA)

## Preprocessing Pipeline

## Correlation & Multicollinearity

## Model Training, Evaluation & Best-Model Justification